# 01.8 — Feature Selection: rdmoldes 50-descriptor set

**Goal**: select a global (endpoint-agnostic) subset of the paper's 50 rdmoldes descriptors
(316 expanded features) for the RF / LightGBM / FCNN data-quantity + noise study.

**Scope**: selection runs on the `rdkit` featureset only (rdmoldes, 316 features) —
FCFP4/ECFP4 fingerprints are a separate featureset and are not touched by this pipeline.

**Granularity**: selection decisions are made per *descriptor* (one of the 50 named
descriptors — 44 single-feature scalars + 6 multi-feature vector descriptors like
`AUTOCORR2D` -> 192 features), never per individual expanded feature. See
`src/feature_selection/`.

**Pipeline**:
1. Drop degenerate (near-constant) descriptors
2. Reduce each surviving descriptor to its top-k principal components (needed to compare
   multi-feature descriptors without collapsing them to one coincidental axis)
3. Mutual information per descriptor, per endpoint -> max across endpoints ("keep if useful for any endpoint")
4. Correlation prune (drop near-duplicate descriptors via canonical correlation, MI tiebreak)
5. VIF prune (drop multicollinear descriptors, MI tiebreak)
6. LightGBM recursive descriptor elimination -> inspect the CV-score trace, pick a cutoff by eye

Selected once globally (all 4 modelling endpoints); revisit per-endpoint only if one endpoint
performs significantly worse after this.

## 0 — Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold, cross_val_score

from src.feature_selection import (
    rdmoldes_descriptor_map,
    drop_constant_descriptors,
    descriptor_component_matrix,
    descriptor_pc1_matrix,
    mutual_info_per_descriptor,
    correlation_prune,
    vif_prune,
    vif_mi_table,
    evaluate_descriptor_set,
    run_descriptor_rfe,
)

RANDOM_STATE = 42
DATA_PROC = Path('../data/processed')
FIGURES = Path('../figures')

ENDPOINTS_MODEL = ['HLM', 'MDR1', 'SOL', 'RLM']  # PPB_H/PPB_R excluded from modelling (too sparse)
N_COMPONENTS = 5  # top-k PCs per descriptor, for MI + correlation-pruning (not VIF, which stays PC1-only)

feat = joblib.load(DATA_PROC / 'section3_feat.pkl')
for ep in ENDPOINTS_MODEL:
    print(f"{ep:6s}  N={feat[ep]['rdkit'].shape[0]:5d}  rdkit_features={feat[ep]['rdkit'].shape[1]}")

# X_by_endpoint/y_by_endpoint_arr: each endpoint's own real rows (no dedup, no pooling) --
# used everywhere a real model actually gets fit (R^2 audits below, RFE in section 5).
X_by_endpoint = {ep: feat[ep]['rdkit'] for ep in ENDPOINTS_MODEL}
y_by_endpoint_arr = {ep: feat[ep]['y'] for ep in ENDPOINTS_MODEL}

## 1 — Descriptor map + constant-descriptor filter

Steps 1-4 (variance, MI, correlation, VIF) only ever look at the descriptor features
themselves (`X`), not any endpoint's target — descriptor redundancy is a property of the
molecules' structures, not of a specific endpoint. Many compounds are tested across more
than one of the 4 endpoints, so naively row-stacking all 4 endpoints' matrices would count
those molecules 2-4x when estimating variance/correlation/VIF, giving them outsized
influence on what "typical" redundancy looks like. Instead we pool the **unique molecules**
only (deduplicated by canonical SMILES) for steps 1-4. Step 6 (RFE) and the MI step still
use each endpoint's own full (non-deduplicated) rows against its own target — that's
real data, not an estimation-bias concern.

A descriptor is "**constant**" if every one of its features has ~zero variance across all
molecules — i.e. it returns essentially the same value for every compound, so it can never
help distinguish one molecule from another.

In [ ]:
# Deduplicate by canonical SMILES across the 4 endpoints -- each unique molecule
# contributes exactly one row to X_all, regardless of how many endpoints tested it.
smiles_to_gid = {}
X_unique_rows = []
endpoint_gids = {}  # ep -> global ids for that endpoint's own rows, in that endpoint's own order
for ep in ENDPOINTS_MODEL:
    gids = np.empty(len(feat[ep]['smiles']), dtype=int)
    for i, smi in enumerate(feat[ep]['smiles']):
        if smi not in smiles_to_gid:
            smiles_to_gid[smi] = len(X_unique_rows)
            X_unique_rows.append(feat[ep]['rdkit'][i])
        gids[i] = smiles_to_gid[smi]
    endpoint_gids[ep] = gids
X_all = np.array(X_unique_rows)

n_endpoint_rows = sum(feat[ep]['rdkit'].shape[0] for ep in ENDPOINTS_MODEL)
print(f'{len(smiles_to_gid)} unique molecules from {n_endpoint_rows} endpoint-rows '
      f'({n_endpoint_rows - len(smiles_to_gid)} cross-endpoint duplicates removed)')
print(f'X_all shape: {X_all.shape}')

descriptor_map = rdmoldes_descriptor_map()
print(f'{len(descriptor_map)} descriptors, {sum(len(f) for f in descriptor_map.values())} total features')

kept_map, dropped_constant = drop_constant_descriptors(X_all, descriptor_map)
print(f'Dropped {len(dropped_constant)} constant descriptor(s): {dropped_constant}')
print(f'{len(kept_map)} descriptors remain')

# CalcPBF and CalcSpherocityIndex are exactly 0.0 for every molecule here (not just
# near-zero) -- other 3D descriptors (CalcAsphericity, CalcEccentricity) vary normally on
# the same conformers, so this isn't a conformer-generation failure, just a quirk specific
# to these two RDKit calls in this environment. Correctly dropped either way; noted here
# in case either descriptor is ever relied on elsewhere in the project.
for name in dropped_constant:
    print(f'  {name}: unique value = {np.unique(X_all[:, descriptor_map[name]])}')

baseline_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, list(kept_map.keys()), random_state=RANDOM_STATE)
print(f'\nStage R^2 audit -- post constant-filter ({len(kept_map)} descriptors): {baseline_r2}')

## 2 — Descriptor components + mutual information

Each descriptor reduces to its top `N_COMPONENTS` principal components (fewer if the
descriptor has fewer features — the 44 scalar descriptors always reduce to 1).
Correlation-pruning and MI use this full component set rather than a single PC1, so a
large descriptor like `AUTOCORR2D` (192 features) isn't judged by one
coincidentally-informative-or-not axis. MI is computed per endpoint (each endpoint has its
own non-NaN row subset), taking the max over (component, endpoint) per descriptor — a
descriptor is kept if it's useful for *any* endpoint.

In [ ]:
component_map = descriptor_component_matrix(X_all, kept_map, n_components=N_COMPONENTS, random_state=RANDOM_STATE)
print(f'{len(component_map)} descriptors; components per descriptor (min/max): '
      f'{min(c.shape[1] for c in component_map.values())}/{max(c.shape[1] for c in component_map.values())}')

# y indexed by each endpoint's own global molecule ids (not a contiguous range) --
# mutual_info_per_descriptor positionally indexes component_map by these, so a molecule
# shared across endpoints correctly reuses the same descriptor row for each endpoint's own MI calc.
y_by_endpoint = {}
for ep in ENDPOINTS_MODEL:
    y_by_endpoint[ep] = pd.Series(feat[ep]['y'], index=endpoint_gids[ep])

mi_max, mi_table = mutual_info_per_descriptor(component_map, y_by_endpoint, random_state=RANDOM_STATE)
mi_table['max'] = mi_max
mi_table['mi_rank'] = mi_max.rank(ascending=False)

print('\nMutual information (nats) per descriptor, per endpoint -- HLM/MDR1/SOL/RLM columns are '
      "MI(descriptor's best component, that endpoint's target); 'max' is the max across the 4 "
      "endpoints (the score used everywhere downstream); 'mi_rank' ranks descriptors by that max, "
      "most useful first.")
mi_table.sort_values('max', ascending=False)

## 3 — Correlation prune

Drop one descriptor from each near-duplicate pair (canonical correlation between their
top-component sets > 0.9), keeping whichever has the higher max-MI.

**Why CCA here and not just PCA again**: PCA (step 2) answers a *within*-descriptor
question — "what are this descriptor's own dominant axes of variation." Canonical
Correlation Analysis (CCA) answers a *between*-descriptor question — "how correlated are
these two descriptors with each other," using all of each one's top components jointly
rather than only comparing their first components to each other. It's the standard
generalisation of Pearson correlation from two single variables to two multi-dimensional
variable sets — exactly what's needed to judge whether two descriptors are redundant
without arbitrarily privileging their dominant axis.

**Known limitation**: CCA only finds the strongest *linear* combination correlation
between two descriptors' components — genuinely nonlinear (but real) dependence between a
pair could be missed, letting a redundant pair survive as "not correlated." A
distance-correlation-style measure (zero if and only if truly independent, linear or not)
would close this gap, at the cost of a new dependency and O(n²) compute over ~3500
molecules. Not adopted here: VIF and the RFE/R² audit downstream both re-check whatever
this step decides against real model performance, so a wrong call here isn't silently
trusted — see `_canonical_correlation`'s docstring in `src/feature_selection/`.

In [ ]:
CORR_THRESHOLD = 0.9

kept_after_corr, corr_dropped = correlation_prune(component_map, mi_max, threshold=CORR_THRESHOLD)
print(f'Dropped {len(corr_dropped)} descriptor(s) on correlation:')
for rec in corr_dropped:
    print(f"  {rec['dropped']:28s} (kept {rec['kept_instead']:28s} instead, canonical_corr={rec['canonical_corr']:.3f})")
print(f'{len(kept_after_corr)} descriptors remain')

corr_stage_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, kept_after_corr, random_state=RANDOM_STATE)
print(f'\nStage R^2 audit -- post correlation-prune ({len(kept_after_corr)} descriptors): {corr_stage_r2}')
print(f'  -> mean R^2: {corr_stage_r2["mean"]:.4f}  (was {baseline_r2["mean"]:.4f} before any pruning)')

## 4 — VIF prune

First, a combined VIF+MI table for every block still standing — so a high VIF and a high
MI can be eyeballed side by side before any automated elimination happens (this is what
`vif_prune` below does automatically, one block at a time, but seeing the whole table at
once makes it easy to sanity-check that automation isn't cutting something important).

Then: iteratively drop the highest-VIF block (multicollinearity against all other
surviving blocks, not just a pairwise check) until every remaining block is at or below
the threshold. Near-tied VIFs are broken by MI.

In [ ]:
pc1_df = descriptor_pc1_matrix(component_map)  # VIF is inherently single-variable-vs-the-rest; stays PC1-only
vif_mi_table(pc1_df[kept_after_corr], mi_max)

In [ ]:
VIF_THRESHOLD = 5.0

# Static mi_rank among the candidate pool VIF starts from (kept_after_corr) -- printed
# alongside each drop below. vif_elim_rank isn't included in that per-drop print: VIF is
# recomputed every iteration as descriptors leave, so "rank" only has a fixed meaning at
# the very start (see the vif_mi_table above for that snapshot) -- past the first
# iteration it would need re-deriving from vif_prune's internals, not worth the complexity.
mi_rank_at_vif_start = mi_max[kept_after_corr].rank(ascending=False)

kept_after_vif, vif_trace = vif_prune(pc1_df[kept_after_corr], mi_max, threshold=VIF_THRESHOLD)
print(f'Dropped {len(vif_trace)} descriptor(s) on VIF:')
for rec in vif_trace:
    print(f"  {rec['dropped']:28s} VIF={rec['vif']:8.2f}  MI={rec['mi']:.4f}  (mi_rank={mi_rank_at_vif_start[rec['dropped']]:.0f}/{len(kept_after_corr)})  ({rec['remaining']} left)")
print(f'{len(kept_after_vif)} descriptors remain: {kept_after_vif}')

vif_stage_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, kept_after_vif, random_state=RANDOM_STATE)
print(f'\nStage R^2 audit -- before VIF (={len(kept_after_corr)}-descriptor post-correlation stage): {corr_stage_r2}')
print(f'Stage R^2 audit -- after VIF  ({len(kept_after_vif)} descriptors):                         {vif_stage_r2}')
print(f'  -> mean R^2: {vif_stage_r2["mean"]:.4f}  (was {corr_stage_r2["mean"]:.4f} after correlation-prune, {baseline_r2["mean"]:.4f} before any pruning)')

## 5 — LightGBM recursive descriptor elimination

Operates on each endpoint's own full features (not PC1) for the surviving descriptors. At
each step, fits LightGBM per endpoint, scores each descriptor by its max feature
gain-importance (max across endpoints too — same "useful for any endpoint" rule), and
eliminates the lowest-scoring descriptor. The CV score is logged at every step so a bad
elimination shows up as a visible drop in the trace rather than being silently accepted.

**`MIN_DESCRIPTORS`**: how far down the elimination is allowed to go before stopping.
Set low (3) so the trace actually reaches a point where performance visibly falls off,
rather than stopping at an arbitrary floor before we know whether it would have kept
holding up — an earlier run used `MIN_BLOCKS=10` picked with no such justification, and
the trace never got far enough to show whether 10 was near a real elbow or just where the
search happened to stop.

**`CV_FOLDS`**: 5, not 3. 3-fold gave visibly noisy, non-monotonic step-to-step swings in
the earlier run (values zigzagging up and down as descriptors were removed, inconsistent
with a real information-loss trend) — 5-fold trades some speed for a less noisy trace,
which matters here since the whole point of this section is reading the trace by eye.

In [ ]:
rfe_descriptor_map = {name: descriptor_map[name] for name in kept_after_vif}

MIN_DESCRIPTORS = 3  # low on purpose -- see markdown above; we want to see the real elbow, not stop before it
CV_FOLDS = 5  # up from 3 -- see markdown above; 3-fold was too noisy to read the trace by eye

rfe_trace = run_descriptor_rfe(
    X_by_endpoint,
    y_by_endpoint_arr,
    rfe_descriptor_map,
    min_descriptors=MIN_DESCRIPTORS,
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
)
joblib.dump(rfe_trace, DATA_PROC / 'section_fs_rfe_trace.pkl')
print(f'RFE trace: {len(rfe_trace)} steps, from {rfe_trace[0]["n_descriptors"]} down to {rfe_trace[-1]["n_descriptors"]} descriptors')

## 6 — Inspect the trace, pick a cutoff

Plot mean CV score (across the 4 endpoints) against the number of surviving descriptors.
Pick the smallest descriptor count before the score visibly drops off.

In [ ]:
# Combine the earlier stage audits (48 post-constant-filter, 27 post-correlation, 16
# post-VIF -- the last of which is also rfe_trace's own first row) with the RFE trace
# (16 -> 3), so the whole pipeline's R^2-vs-descriptors story is one continuous view
# rather than only the RFE sub-range.
combined = [
    {'n_descriptors': len(kept_map), 'cv_score_mean': baseline_r2['mean'], 'stage': 'post constant-filter'},
    {'n_descriptors': len(kept_after_corr), 'cv_score_mean': corr_stage_r2['mean'], 'stage': 'post correlation-prune (CCA)'},
] + [{'n_descriptors': row['n_descriptors'], 'cv_score_mean': row['cv_score_mean'], 'stage': 'RFE'} for row in rfe_trace]

for row in combined:
    print(f"{row['n_descriptors']:3d} descriptors  mean_r2={row['cv_score_mean']:.4f}  ({row['stage']})")

n_descriptors_seq = [row['n_descriptors'] for row in combined]
score_seq = [row['cv_score_mean'] for row in combined]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_descriptors_seq, score_seq, marker='o')
ax.axvline(len(kept_after_corr), color='tab:orange', linestyle=':', linewidth=1.5,
           label=f'canonical-correlation prune (CCA): 48 -> {len(kept_after_corr)}')
ax.axvline(len(kept_after_vif), color='grey', linestyle='--', linewidth=1,
           label=f'VIF prune: {len(kept_after_corr)} -> {len(kept_after_vif)} (RFE starts here)')
ax.axvline(5, color='tab:green', linestyle=':', linewidth=1.5,
           label='selected cutoff: N=5 (elbow -- see section 6)')
ax.set_xlabel('Number of descriptors remaining')
ax.set_ylabel('Mean CV R² across endpoints')
ax.set_title('R² vs. descriptors remaining — full pipeline (constant-filter -> CCA correlation-prune -> VIF -> RFE)')
ax.invert_xaxis()
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'section_fs_rfe_trace.png', dpi=150)
plt.show()

In [ ]:
# Chosen after inspecting the trace above: R^2 declines gently and roughly monotonically
# from 16 down to 5 descriptors (each step <=0.015), then hits a real elbow at 5->4
# (-0.076 dropping SlogP_VSA -- roughly 5x any other single-step change in the trace).
# 5 descriptors is the smallest set before that cliff.
SELECTED_N_DESCRIPTORS = 5

chosen_row = next(row for row in rfe_trace if row['n_descriptors'] == SELECTED_N_DESCRIPTORS)
selected_descriptors = chosen_row['descriptors']
selected_features = sorted(f for name in selected_descriptors for f in descriptor_map[name])

print(f'Selected {len(selected_descriptors)} descriptors -> {len(selected_features)} of 316 rdmoldes features:')
print(f'  {selected_descriptors}')

print(f'\nFor reference -- the {len(kept_after_corr)} descriptors after correlation-prune (before VIF/RFE):')
print(f'  {kept_after_corr}')

print(f'\nCV R^2 by stage:')
print(f'  {len(kept_map):3d} descriptors, no pruning yet:                    {baseline_r2["mean"]:.4f}')
print(f'  {len(kept_after_corr):3d} descriptors, after canonical-correlation prune (CCA): {corr_stage_r2["mean"]:.4f}')
print(f'  {len(kept_after_vif):3d} descriptors, after VIF prune:                          {vif_stage_r2["mean"]:.4f}')
print(f'  {SELECTED_N_DESCRIPTORS:3d} descriptors, after LightGBM RFE (final cutoff):    {chosen_row["cv_score_mean"]:.4f}')

output = {
    'selected_descriptors': selected_descriptors,
    'selected_features': selected_features,
    'cv_score_mean': chosen_row['cv_score_mean'],
    'cv_score_by_endpoint': chosen_row['cv_score_by_endpoint'],
    'baseline_cv_score_mean': baseline_r2['mean'],
    'post_cca_correlation_prune_cv_score_mean': corr_stage_r2['mean'],
    'post_vif_prune_cv_score_mean': vif_stage_r2['mean'],
    'dropped_constant': dropped_constant,
    'dropped_correlation': corr_dropped,
    'dropped_vif': vif_trace,
}
with open(DATA_PROC / 'selected_descriptors.json', 'w') as f:
    json.dump(output, f, indent=2)
print(f'\nSaved -> {DATA_PROC / "selected_descriptors.json"}')

## 7 — Diagnostic: does real 3D geometry matter? (ADR-011 follow-up)

**ADR-011** (`DECISIONS.md`) found that 11 of `rdmoldes()`'s "3D shape" descriptors are
computed on the flat 2D depiction coordinates in the source SDFs (`Is3D()=False`, all
`z=0`), not real conformers — inherited from the paper's own methodology. Its original
"limited impact" claim was wrong: `CalcEccentricity` and `CalcPMI3` are 2 of the 5
descriptors in this notebook's final selection (§6).

This section is the cheap diagnostic ADR-011 calls for: embed **real** 3D conformers for
the same 3509 unique molecules already used above, recompute just the 9 affected
descriptors that survived the constant-filter (`CalcPBF`/`CalcSpherocityIndex` stay
excluded — they were dropped for being exactly constant on the flat data, not part of this
comparison), splice them into the existing feature matrices in place of the flat values,
and re-run `evaluate_descriptor_set` — no `01.5` retraining, no re-running the pruning
pipeline itself. If CV R² doesn't move, that's a clean negative result; if it does, a full
propagation into `01.5` (ADR-011 Step 2) is warranted.

**Conformer choice**: several of the affected descriptors (`CalcAsphericity`,
`CalcEccentricity`, `CalcPMI1-3`, `CalcRadiusOfGyration`, `CalcInertialShapeFactor`) can
vary across a flexible molecule's accessible conformers — a single arbitrarily-seeded
embedding isn't a representative "real 3D geometry" for such a molecule, only one
snapshot of it. So this section embeds an **ensemble of `N_CONFORMERS` conformers per
molecule** (ETKDGv3, fixed base seed for reproducibility) and computes descriptors on the
**lowest-MMFF-energy** conformer of each — the standard cheap proxy for "the geometry a
molecule is actually most likely to be found in," rather than one arbitrary embedding.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors

# All 11 individual flat-geometry descriptors named in ADR-011 (CalcPBF/CalcSpherocityIndex
# included for completeness even though they're already excluded from every set evaluated
# below -- constant-filtered on the flat data and never reconsidered here).
AFFECTED_3D_DESCRIPTORS = [
    'CalcPBF', 'CalcSpherocityIndex', 'CalcNPR1', 'CalcNPR2',
    'CalcPMI1', 'CalcPMI2', 'CalcPMI3',
    'CalcAsphericity', 'CalcEccentricity',
    'CalcRadiusOfGyration', 'CalcInertialShapeFactor',
]

N_CONFORMERS = 5  # ensemble size per molecule -- see markdown above for why a single embedding isn't enough
MMFF_MAX_ITERS = 500


def _compute_3d_descriptors(mol, conf_id):
    return {
        'CalcPBF': rdMolDescriptors.CalcPBF(mol, confId=conf_id),
        'CalcSpherocityIndex': rdMolDescriptors.CalcSpherocityIndex(mol, confId=conf_id),
        'CalcNPR1': rdMolDescriptors.CalcNPR1(mol, confId=conf_id),
        'CalcNPR2': rdMolDescriptors.CalcNPR2(mol, confId=conf_id),
        'CalcPMI1': rdMolDescriptors.CalcPMI1(mol, confId=conf_id),
        'CalcPMI2': rdMolDescriptors.CalcPMI2(mol, confId=conf_id),
        'CalcPMI3': rdMolDescriptors.CalcPMI3(mol, confId=conf_id),
        'CalcAsphericity': rdMolDescriptors.CalcAsphericity(mol, confId=conf_id),
        'CalcEccentricity': rdMolDescriptors.CalcEccentricity(mol, confId=conf_id),
        'CalcRadiusOfGyration': rdMolDescriptors.CalcRadiusOfGyration(mol, confId=conf_id),
        'CalcInertialShapeFactor': rdMolDescriptors.CalcInertialShapeFactor(mol, confId=conf_id),
    }


def _embed_lowest_energy_conformer(smiles, seed, num_confs=N_CONFORMERS, max_iters=MMFF_MAX_ITERS, max_retries=5):
    """Embed an ensemble of conformers (ETKDGv3) and return (mol, lowest-MMFF-energy confId).

    A single arbitrarily-seeded embedding is only one snapshot of a flexible molecule's
    accessible geometries -- num_confs embeddings, MMFF-optimized, keeping the lowest-energy
    one, is the standard cheap proxy for "the geometry the molecule is actually most likely
    to be found in" rather than an arbitrary one. Falls back across a few base seeds if
    embedding produces zero conformers; returns (None, None) if it never succeeds.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.useRandomCoords = True
    cids = []
    for attempt in range(max_retries):
        params.randomSeed = seed + attempt
        cids = list(AllChem.EmbedMultipleConfs(mol_h, numConfs=num_confs, params=params))
        if cids:
            break
    if not cids:
        return None, None
    try:
        energies = AllChem.MMFFOptimizeMoleculeConfs(mol_h, maxIters=max_iters)
        converged = [(cid, e) for cid, (not_converged, e) in zip(cids, energies) if not_converged == 0]
        candidates = converged if converged else list(zip(cids, [e for _, e in energies]))
        best_cid = min(candidates, key=lambda t: t[1])[0]
    except Exception:
        best_cid = cids[0]  # optimization failed outright -- fall back to the first embedded conformer
    return mol_h, best_cid


# unique_smiles ordered by gid, matching X_all's row order from section 1
unique_smiles = [None] * len(smiles_to_gid)
for smi, gid in smiles_to_gid.items():
    unique_smiles[gid] = smi

new_desc_by_smiles = {}
embedding_failures = []
for smi in unique_smiles:
    mol_h, best_cid = _embed_lowest_energy_conformer(smi, seed=RANDOM_STATE)
    if mol_h is None:
        embedding_failures.append(smi)
        continue
    new_desc_by_smiles[smi] = _compute_3d_descriptors(mol_h, best_cid)

print(f'Embedded {N_CONFORMERS}-conformer ensembles for {len(new_desc_by_smiles)}/{len(unique_smiles)} unique molecules, '
      f'descriptors computed on each one\'s lowest-MMFF-energy conformer '
      f'({len(embedding_failures)} embedding failures -- their original flat-derived values are kept as fallback, not dropped)')

In [ ]:
# Splice the recomputed values into each endpoint's own feature matrix (X_by_endpoint --
# real per-endpoint rows, not the deduplicated X_all), in place of the flat-derived values,
# for the descriptors that actually survived the constant-filter (kept_map). Molecules
# with a failed embedding keep their original flat value untouched.
descriptors_to_splice = [d for d in AFFECTED_3D_DESCRIPTORS if d in kept_map]
print(f'Splicing {len(descriptors_to_splice)} descriptors that survived the constant-filter: {descriptors_to_splice}')

X_by_endpoint_3d = {ep: X.copy() for ep, X in X_by_endpoint.items()}
for ep in ENDPOINTS_MODEL:
    for i, smi in enumerate(feat[ep]['smiles']):
        if smi not in new_desc_by_smiles:
            continue  # embedding failed for this molecule -- flat value stays
        new_vals = new_desc_by_smiles[smi]
        for name in descriptors_to_splice:
            col = descriptor_map[name][0]  # each is a 1-feature scalar descriptor
            X_by_endpoint_3d[ep][i, col] = new_vals[name]

In [ ]:
# Re-run evaluate_descriptor_set with real-conformer values spliced in, at the two stages
# where the affected descriptors are actually present: the final 5-descriptor selection
# (CalcEccentricity, CalcPMI3) and the 48-descriptor post-constant-filter baseline (9 of
# the 11 affected descriptors, everything but the already-dropped PBF/SpherocityIndex).
selection_r2_3d = evaluate_descriptor_set(X_by_endpoint_3d, y_by_endpoint_arr, descriptor_map, selected_descriptors, random_state=RANDOM_STATE)
baseline_r2_3d = evaluate_descriptor_set(X_by_endpoint_3d, y_by_endpoint_arr, descriptor_map, list(kept_map.keys()), random_state=RANDOM_STATE)

print(f'Final {SELECTED_N_DESCRIPTORS}-descriptor selection {selected_descriptors}:')
print(f'  flat (original):  mean R^2 = {chosen_row["cv_score_mean"]:.4f}  {chosen_row["cv_score_by_endpoint"]}')
print(f'  real conformers:  mean R^2 = {selection_r2_3d["mean"]:.4f}  { {k: v for k, v in selection_r2_3d.items() if k != "mean"} }')
print(f'  delta: {selection_r2_3d["mean"] - chosen_row["cv_score_mean"]:+.4f}')

print(f'\n{len(kept_map)}-descriptor post-constant-filter baseline:')
print(f'  flat (original):  mean R^2 = {baseline_r2["mean"]:.4f}  { {k: v for k, v in baseline_r2.items() if k != "mean"} }')
print(f'  real conformers:  mean R^2 = {baseline_r2_3d["mean"]:.4f}  { {k: v for k, v in baseline_r2_3d.items() if k != "mean"} }')
print(f'  delta: {baseline_r2_3d["mean"] - baseline_r2["mean"]:+.4f}')

diagnostic_output = {
    'n_unique_molecules': len(unique_smiles),
    'n_embedding_failures': len(embedding_failures),
    'descriptors_spliced': descriptors_to_splice,
    'selection_5desc': {
        'descriptors': selected_descriptors,
        'flat_cv_score_mean': chosen_row['cv_score_mean'],
        'flat_cv_score_by_endpoint': chosen_row['cv_score_by_endpoint'],
        'real_conformer_cv_score_mean': selection_r2_3d['mean'],
        'real_conformer_cv_score_by_endpoint': {k: v for k, v in selection_r2_3d.items() if k != 'mean'},
        'delta': selection_r2_3d['mean'] - chosen_row['cv_score_mean'],
    },
    'baseline_48desc': {
        'descriptors': list(kept_map.keys()),
        'flat_cv_score_mean': baseline_r2['mean'],
        'flat_cv_score_by_endpoint': {k: v for k, v in baseline_r2.items() if k != 'mean'},
        'real_conformer_cv_score_mean': baseline_r2_3d['mean'],
        'real_conformer_cv_score_by_endpoint': {k: v for k, v in baseline_r2_3d.items() if k != 'mean'},
        'delta': baseline_r2_3d['mean'] - baseline_r2['mean'],
    },
}
with open(DATA_PROC / 'adr011_3d_diagnostic.json', 'w') as f:
    json.dump(diagnostic_output, f, indent=2)
print(f'\nSaved -> {DATA_PROC / "adr011_3d_diagnostic.json"}')

In [ ]:
# Report-ready table: flat vs. real-conformer CV R^2, both descriptor sets, per endpoint + mean.
selection_flat_scores = dict(chosen_row['cv_score_by_endpoint'], mean=chosen_row['cv_score_mean'])

comparison_rows = []
for set_label, flat_scores, real_scores in [
    (f'Final selection ({SELECTED_N_DESCRIPTORS} descriptors)', selection_flat_scores, selection_r2_3d),
    (f'Post-constant-filter baseline ({len(kept_map)} descriptors)', baseline_r2, baseline_r2_3d),
]:
    for ep in ENDPOINTS_MODEL + ['mean']:
        flat_val = flat_scores[ep]
        real_val = real_scores[ep]
        comparison_rows.append({
            'descriptor_set': set_label,
            'endpoint': ep,
            'flat_r2': flat_val,
            'real_conformer_r2': real_val,
            'delta': real_val - flat_val,
        })
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(DATA_PROC / 'adr011_3d_diagnostic_comparison.csv', index=False)
comparison_df

In [ ]:
# Report-ready table: R^2 cost of each pruning stage, for direct comparison against the
# real-conformer delta above (both final-selection and 48-descriptor-baseline deltas are
# smaller than every single stage's own cost below).
stage_cost_rows = [
    {'stage': 'Post constant-filter (start)', 'n_descriptors': len(kept_map), 'mean_cv_r2': baseline_r2['mean'], 'delta_from_previous_stage': None},
    {'stage': 'Post correlation-prune (CCA)', 'n_descriptors': len(kept_after_corr), 'mean_cv_r2': corr_stage_r2['mean'], 'delta_from_previous_stage': corr_stage_r2['mean'] - baseline_r2['mean']},
    {'stage': 'Post VIF-prune', 'n_descriptors': len(kept_after_vif), 'mean_cv_r2': vif_stage_r2['mean'], 'delta_from_previous_stage': vif_stage_r2['mean'] - corr_stage_r2['mean']},
    {'stage': 'Post LightGBM RFE (final cutoff)', 'n_descriptors': SELECTED_N_DESCRIPTORS, 'mean_cv_r2': chosen_row['cv_score_mean'], 'delta_from_previous_stage': chosen_row['cv_score_mean'] - vif_stage_r2['mean']},
    {'stage': '(for reference) real-conformer splice, final selection', 'n_descriptors': SELECTED_N_DESCRIPTORS, 'mean_cv_r2': selection_r2_3d['mean'], 'delta_from_previous_stage': selection_r2_3d['mean'] - chosen_row['cv_score_mean']},
]
stage_cost_df = pd.DataFrame(stage_cost_rows)
stage_cost_df.to_csv(DATA_PROC / 'section_fs_pruning_stage_costs.csv', index=False)
stage_cost_df

## 8 — Diagnostic: gain-importance shadowing/masking

`run_descriptor_rfe` (§5) eliminates using LightGBM gain-importance — how often a
descriptor gets used for a split. This is a known-imperfect proxy when correlated
descriptors compete: gradient-boosted trees tend to pick *one* of a redundant group and
give it all the credit, leaving an equally-capable alternative looking worthless. That's
not hypothetical here: `CalcEccentricity` (final selection, §6) has **zero** LightGBM
gain-importance in the full 316-feature model, across all 4 endpoints, while its closest
relative `CalcAsphericity` (dropped at VIF, §4) has real, nonzero importance there — yet
`CalcEccentricity` survived RFE to the final cutoff and Asphericity didn't.

Three checks, generalised across the whole §4 candidate pool (not hardcoded to this one
pair) so this is a reusable diagnostic, not a one-off:

1. **Rank-correlation redundancy CCA/VIF may have missed** — CCA (§3) and VIF (§4) are
   both fundamentally *linear* checks. Two descriptors can be only moderately correlated
   linearly (canonical r < 0.9, passing both checks) while being **exactly rank-identical**
   (Spearman ρ=1.0) — informationally indistinguishable to any *tree* model (which only
   ever asks "which side of a threshold"), even though a smooth model like FCNN could
   still tell them apart.
2. **RF vs. LightGBM importance divergence** — do the two tree ensembles even agree on
   which descriptors matter, in the full 316-feature set? If not, LightGBM-only RFE is a
   biased view of "what these models find important" (FCNN excluded from this specific
   check — no native importance metric, would need permutation importance and a trained
   model; left for when FCNN is actually built for the quantity/noise experiments).
3. **Direct swap test** — for any §6-selected descriptor with a near-rank-identical
   relative among the descriptors §3/§4 dropped, does swapping one for the other change CV
   R² at all? A near-zero delta confirms the descriptor's presence in the final set is
   substitutable, not uniquely load-bearing.

In [ ]:
from scipy.stats import spearmanr

# For every descriptor still in the final selection, find its most rank-correlated
# relative among ALL earlier-dropped scalar descriptors (constant/correlation/VIF/RFE),
# using the deduplicated pool X_all. Only scalar (1-feature) descriptors are checked here
# -- rank correlation between a scalar and a multi-feature vector descriptor isn't a
# single well-defined number the way it is between two scalars.
all_dropped_names = dropped_constant + [r['dropped'] for r in corr_dropped] + [r['dropped'] for r in vif_trace] + \
    [name for name in kept_after_vif if name not in selected_descriptors]
scalar_dropped = [name for name in all_dropped_names if len(descriptor_map[name]) == 1]

shadow_candidates = {}
for name in selected_descriptors:
    if len(descriptor_map[name]) != 1:
        continue  # skip multi-feature descriptors (PEOE_VSA, SlogP_VSA) -- same reasoning as above
    target = X_all[:, descriptor_map[name][0]]
    best_rho, best_partner = 0.0, None
    for other in scalar_dropped:
        rho, _ = spearmanr(target, X_all[:, descriptor_map[other][0]])
        if abs(rho) > abs(best_rho):
            best_rho, best_partner = rho, other
    shadow_candidates[name] = (best_partner, best_rho)
    flag = ' <-- NEAR RANK-IDENTICAL (tree-invisible redundancy)' if abs(best_rho) > 0.95 else ''
    print(f'{name:20s} most rank-correlated with {best_partner or "(none found)":26s}  Spearman rho={best_rho:+.4f}{flag}')

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Fit LightGBM and RF on the FULL 316-feature set (not the pruned selection) for each
# endpoint, then compare importance for every final-selection descriptor plus its shadow
# candidate identified above -- shows whether the two model types even agree on which
# descriptors matter, before any pruning has happened.
compare_names = sorted(set(selected_descriptors) | {p for p, _ in shadow_candidates.values() if p is not None})

lgbm_imp_by_ep, rf_imp_by_ep = {}, {}
for ep in ENDPOINTS_MODEL:
    X = feat[ep]['rdkit']
    y = feat[ep]['y']
    lgbm = LGBMRegressor(n_estimators=300, random_state=RANDOM_STATE, verbose=-1).fit(X, y)
    rf = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(X, y)
    lgbm_imp_by_ep[ep] = lgbm.feature_importances_ / lgbm.feature_importances_.sum()
    rf_imp_by_ep[ep] = rf.feature_importances_ / rf.feature_importances_.sum()

# Mean normalised importance across the 4 endpoints, single-feature descriptors only
# (same reasoning as the shadow-candidate search -- a multi-feature descriptor's
# importance would need summing across its own features to compare fairly, skip for now).
lgbm_mean = {name: np.mean([lgbm_imp_by_ep[ep][descriptor_map[name][0]] for ep in ENDPOINTS_MODEL])
             for name in compare_names if len(descriptor_map[name]) == 1}
rf_mean = {name: np.mean([rf_imp_by_ep[ep][descriptor_map[name][0]] for ep in ENDPOINTS_MODEL])
           for name in compare_names if len(descriptor_map[name]) == 1}

plot_names = list(lgbm_mean.keys())
x = np.arange(len(plot_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars_lgbm = ax.bar(x - width/2, [lgbm_mean[n] for n in plot_names], width, label='LightGBM', color='tab:blue')
bars_rf = ax.bar(x + width/2, [rf_mean[n] for n in plot_names], width, label='RandomForest', color='tab:orange')
ax.set_xticks(x)
ax.set_xticklabels(plot_names, rotation=30, ha='right')
ax.set_ylabel('Mean normalised feature importance (share of total, 4-endpoint average)')
ax.set_title('LightGBM vs. RF importance, full 316-feature model — final-selection descriptors + shadow candidates')
for name in plot_names:
    if name in selected_descriptors:
        ax.get_xticklabels()[plot_names.index(name)].set_fontweight('bold')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'section_fs_importance_shadowing.png', dpi=150)
plt.show()

print('(bold x-labels = in the final 5-descriptor selection; others are their shadow candidates)')
pd.DataFrame({'lightgbm_importance': lgbm_mean, 'rf_importance': rf_mean})

In [ ]:
# For every final-selection descriptor flagged as near-rank-identical (|rho|>0.95) to a
# dropped relative, swap it out for that relative and re-measure CV R^2 on the final
# selection -- a near-zero delta confirms the descriptor is substitutable, not uniquely
# load-bearing (as opposed to e.g. SlogP_VSA, whose removal cost -0.076 in the RFE trace, §6).
final_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, selected_descriptors, random_state=RANDOM_STATE)
print(f'Baseline -- final {len(selected_descriptors)}-descriptor selection: mean R^2 = {final_r2["mean"]:.4f}\n')

swap_results = []
for name, (partner, rho) in shadow_candidates.items():
    if partner is None or abs(rho) <= 0.95:
        continue
    swapped_set = [partner if d == name else d for d in selected_descriptors]
    swapped_r2 = evaluate_descriptor_set(X_by_endpoint, y_by_endpoint_arr, descriptor_map, swapped_set, random_state=RANDOM_STATE)
    delta = swapped_r2['mean'] - final_r2['mean']
    swap_results.append({'original': name, 'swapped_in': partner, 'spearman_rho': rho,
                          'r2_original': final_r2['mean'], 'r2_swapped': swapped_r2['mean'], 'delta': delta})
    print(f'{name} -> {partner}  (rho={rho:+.4f}):  R^2 {final_r2["mean"]:.4f} -> {swapped_r2["mean"]:.4f}  (delta={delta:+.4f})')

if not swap_results:
    print('No final-selection descriptor has a |rho|>0.95 dropped relative -- nothing to swap-test.')
else:
    swap_df = pd.DataFrame(swap_results)
    swap_df.to_csv(DATA_PROC / 'section_fs_shadowing_swap_test.csv', index=False)
    print(f'\nSaved -> {DATA_PROC / "section_fs_shadowing_swap_test.csv"}')

## 9 — Takeaway

**Is there a small, stable descriptor set that gets you most of the full-featureset
performance?** Yes:

| Descriptors | Features | Mean CV R² | % of full-featureset (48-descriptor) performance |
|---|---|---|---|
| 48 (no pruning) | 314 | 0.4193 | 100% |
| 16 (post VIF) | 40 | 0.3824 | 91% |
| 5 (final RFE cutoff) | 29 | 0.3555 | 85% |

**But don't over-read the specific identity of the final 5.** §8 showed `CalcEccentricity`
is rank-identical (Spearman ρ=1.0) to `CalcAsphericity` (dropped at VIF for unrelated
reasons), gets zero LightGBM importance in the full model despite RF finding it
comparable to Asphericity, and swapping it out changes the final selection's CV R² by less
than a rounding error. Its survival through §5's LightGBM-gain-based RFE reflects
elimination order and a single model's importance quirks more than genuine irreplaceable
value.

**Recommendation for the RF/LightGBM/FCNN quantity/noise experiments: use the 16-descriptor
set (§4's `kept_after_vif`), not the 5-descriptor set.** It keeps 91% (not 85%) of
baseline performance, is still an 8x reduction from the original 314 features, and avoids
building the experiments on a descriptor now known to be fragile/substitutable. The
5-descriptor set remains useful as the RFE-trace endpoint that revealed the real elbow
(§6) and as the input to §7/§8's diagnostics, but isn't the recommended production choice.